# Donations — estimated gift value (IS 455 full pipeline)

**Dataset:** `datasets/donations.csv`  
**Run this notebook with working directory:** `ml-pipelines/` (same as other course notebooks).

This notebook is organized to match the course submission checklist:

1. Problem framing  
2. Data acquisition, preparation & exploration  
3. Modeling & feature selection (interpretable + stronger learner)  
4. Evaluation & interpretation (including error costs)  
5. Causal / relationship analysis (honest limits)  
6. Deployment notes (repo integration)  

Additional **textbook-aligned blocks** (Ch. 2–3, 6–8, 9–11, 13, 15–16) appear as subsections: data quality, plots, baseline vs model, residuals, time-based validation, classification metrics, permutation importance, and a **Ch. 1–17 crosswalk** before the causal discussion.


## 1. Problem framing

**Business question:** Development and communications leadership need to understand **what kinds of gifts tend to carry higher normalized value** (`estimated_value`) and whether **channel and campaign** patterns can support budgeting and channel-investment decisions.

**Stakeholders:** Fundraising / development staff, leadership reporting on revenue mix.

**Explanatory goal:** Quantify **associations** between gift type, recurrence, campaign, and channel with value—using interpretable linear models. Coefficients describe patterns in *this* historical data, not guaranteed causal impacts of changing a channel.

**Predictive goal:** Support **rough planning** (ranges, prioritization), not exact donor-level promises. Out-of-sample **MAE** matters; we compare a linear model to a random forest via `pipeline_kit`.

**Why both:** Strategy teams need **interpretable drivers**; operations need **stable forecasts** for scenarios.

**Success metrics:** MAE and $R^2$ on a holdout set for prediction; plausibility and stability of linear associations for explanation.

**Important:** We **do not** use `amount` as a predictor when explaining `estimated_value`—for monetary rows it is essentially the same number (data leakage). Models use metadata (type, channel, campaign, dates, referral flag) instead.


In [46]:
import sys
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import accuracy_score, mean_absolute_error, r2_score, roc_auc_score, roc_curve
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, export_text, plot_tree

NOTEBOOK_DIR = Path.cwd().resolve()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from pipeline_kit import NotebookConfig, datasets_dir, run_notebook_driver

DATA_PATH = datasets_dir() / "donations.csv"
df = pd.read_csv(DATA_PATH)
print("shape:", df.shape)
print("\nColumns:", list(df.columns))
print("\nMissing (selected):")
print(df[["estimated_value", "is_recurring", "campaign_name", "channel_source", "amount"]].isna().sum())
df.head(8)


shape: (420, 13)

Columns: ['donation_id', 'supporter_id', 'donation_type', 'donation_date', 'is_recurring', 'campaign_name', 'channel_source', 'currency_code', 'amount', 'estimated_value', 'impact_unit', 'notes', 'referral_post_id']

Missing (selected):
estimated_value      0
is_recurring         0
campaign_name      275
channel_source       0
amount             186
dtype: int64


,donation_id,supporter_id,donation_type,donation_date,is_recurring,campaign_name,channel_source,currency_code,amount,estimated_value,impact_unit,notes,referral_post_id
0,1,42,Monetary,2025-12-31,False,NaN,Campaign,PHP,717.18,717.18,pesos,In support of safehouse operations,NaN
1,2,25,Time,2025-12-02,True,Year-End Hope,Event,NaN,NaN,35.15,hours,Community outreach support,NaN
2,3,19,Monetary,2024-12-02,False,NaN,PartnerReferral,PHP,1074.65,1074.65,pesos,Campaign support,NaN
3,4,33,Monetary,2023-09-11,False,NaN,PartnerReferral,PHP,1230.56,1230.56,pesos,In support of safehouse operations,NaN
4,5,24,InKind,2023-11-08,True,GivingTuesday,SocialMedia,NaN,NaN,1177.41,items,In support of safehouse operations,421.0
5,6,25,Monetary,2025-09-18,True,NaN,Direct,PHP,678.86,678.86,pesos,Campaign support,NaN
6,7,25,Time,2024-12-14,True,Year-End Hope,Event,NaN,NaN,37.53,hours,Campaign support,NaN
7,8,23,Monetary,2024-11-21,False,GivingTuesday,Direct,PHP,600.64,600.64,pesos,Campaign support,NaN


## 2. Data acquisition, preparation & exploration

Each row is one recorded gift. `estimated_value` is the outcome used for modeling (monetary, in-kind, and time gifts are on a common scale in the dataset). `campaign_name` is often missing— we impute a sentinel in preprocessing.

**Join logic:** Single-table pipeline; no joins in this notebook. (If you later join `supporters` or `social_media_posts`, document keys and avoid future information in features.)


In [47]:
# Distributions and group means (exploration)
print("estimated_value summary:")
print(df["estimated_value"].describe())

q1 = df["estimated_value"].quantile(0.25)
q3 = df["estimated_value"].quantile(0.75)
iqr = q3 - q1
low = q1 - 1.5 * iqr
high = q3 + 1.5 * iqr
outlier_count = ((df["estimated_value"] < low) | (df["estimated_value"] > high)).sum()
print("\nIQR outlier bounds:", round(float(low), 2), "to", round(float(high), 2), "| count:", int(outlier_count))

print("\nMean value by donation_type:")
print(df.groupby("donation_type")["estimated_value"].mean().sort_values(ascending=False))

print("\nMean value by channel_source:")
print(df.groupby("channel_source")["estimated_value"].mean().sort_values(ascending=False))

# Leakage check: amount vs estimated_value (monetary)
if df["amount"].notna().any():
    sub = df[df["donation_type"].eq("Monetary") & df["amount"].notna()]
    if len(sub):
        print("\nMonetary rows: correlation(amount, estimated_value) =", round(float(sub["amount"].corr(sub["estimated_value"])), 4))


estimated_value summary:
count     420.000000
mean      699.304310
std       713.251586
min         2.200000
25%       300.000000
50%       514.160000
75%       989.722500
max      6481.540000
Name: estimated_value, dtype: float64

IQR outlier bounds: -734.58 to 2024.31 | count: 21

Mean value by donation_type:
donation_type
Monetary       1028.737350
InKind          527.738878
Time             19.073696
Skills           12.283158
SocialMedia       6.699565
Name: estimated_value, dtype: float64

Mean value by channel_source:
channel_source
Campaign           770.961176
SocialMedia        702.460897
PartnerReferral    669.468077
Direct             663.681951
Event              650.980000
Name: estimated_value, dtype: float64

Monetary rows: correlation(amount, estimated_value) = 1.0


### Textbook Ch. 2–3: Data quality

Row-level checks before modeling: identifiers, dates, impossible values, and missingness that affects interpretation.


In [48]:
_dq = df.copy()
_dq["donation_date"] = pd.to_datetime(_dq["donation_date"], errors="coerce")
print("Duplicate donation_id:", int(_dq["donation_id"].duplicated().sum()))
print("Rows with missing donation_date:", int(_dq["donation_date"].isna().sum()))
print("Date range:", _dq["donation_date"].min(), "→", _dq["donation_date"].max())
print("Negative estimated_value:", int((_dq["estimated_value"] < 0).sum()))
print("Missing campaign_name (share):", round(_dq["campaign_name"].isna().mean(), 3))
print("Unique donation_type:", _dq["donation_type"].nunique(), "| channel_source:", _dq["channel_source"].nunique())


Duplicate donation_id: 0
Rows with missing donation_date: 0
Date range: 2023-01-09 00:00:00 → 2026-03-01 00:00:00
Negative estimated_value: 0
Missing campaign_name (share): 0.655
Unique donation_type: 5 | channel_source: 5


### Textbook Ch. 6–8: Visual exploration

Distributional shape and category overlap motivate the heavy tails and mixed gift types in `estimated_value`.


In [49]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(df["estimated_value"], bins=30, color="steelblue", edgecolor="white")
axes[0].set_xlabel("estimated_value")
axes[0].set_ylabel("count")
axes[0].set_title("Histogram (Ch. 6)")

_dfv = df.copy()
_dfv.boxplot(column="estimated_value", by="donation_type", ax=axes[1])
axes[1].set_title("Value by donation_type")
axes[1].set_xlabel("donation_type")
plt.suptitle("")
plt.tight_layout()
plt.close()

_num = df[["estimated_value"]].copy()
for _c in ["is_recurring"]:
    if _c in df.columns:
        _num[_c] = df[_c].astype(str).str.lower().eq("true").astype(float)
if "referral_post_id" in df.columns:
    _num["has_referral"] = df["referral_post_id"].notna().astype(float)
print("\nNumeric correlation matrix (engineered + target subset):")
print(_num.corr(numeric_only=True).round(3))



Numeric correlation matrix (engineered + target subset):
                 estimated_value  is_recurring  has_referral
estimated_value            1.000        -0.034         0.004
is_recurring              -0.034         1.000        -0.058
has_referral               0.004        -0.058         1.000


## 3. Modeling — compact interpretable block (manual)

We fit **OLS**, **OLS with robust SE (HC1)**, **Ridge**, a **high-value logistic** model (median split), and a **shallow tree** for readable rules. Features: recurrence, campaign, channel (same spirit as your original notebook, without `amount`).


In [50]:
# Reproducible prep (subset of columns for clear tables)
prep_df = df.drop_duplicates().copy()
prep_df = prep_df.dropna(subset=["estimated_value"]).copy()
prep_df["is_recurring_num"] = prep_df["is_recurring"].astype(str).str.lower().eq("true").astype(int)

X_raw = prep_df[["is_recurring_num", "campaign_name", "channel_source"]].copy()
y = prep_df["estimated_value"].astype(float)

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=42
)

cat_cols = ["campaign_name", "channel_source"]
num_cols = ["is_recurring_num"]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            Pipeline(
                [
                    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
                    ("onehot", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False)),
                ]
            ),
            cat_cols,
        ),
        ("num", "passthrough", num_cols),
    ]
)

X_train_t = preprocessor.fit_transform(X_train_raw)
X_test_t = preprocessor.transform(X_test_raw)
feature_names = list(preprocessor.get_feature_names_out())
X_train_df = pd.DataFrame(X_train_t, columns=feature_names, index=X_train_raw.index)
X_test_df = pd.DataFrame(X_test_t, columns=feature_names, index=X_test_raw.index)

X_train_df = sm.add_constant(X_train_df, has_constant="add")
X_test_df = sm.add_constant(X_test_df, has_constant="add")

ols = sm.OLS(y_train, X_train_df).fit()
print(ols.summary())


                            OLS Regression Results                            
Dep. Variable:        estimated_value   R-squared:                       0.021
Model:                            OLS   Adj. R-squared:                 -0.006
Method:                 Least Squares   F-statistic:                    0.7628
Date:                Wed, 08 Apr 2026   Prob (F-statistic):              0.651
Time:                        16:51:32   Log-Likelihood:                -2635.2
No. Observations:                 336   AIC:                             5290.
Df Residuals:                     326   BIC:                             5328.
Df Model:                           9                                         
Covariance Type:            nonrobust                                         
                                          coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------
co

In [51]:
coef_table = pd.DataFrame({"coef": ols.params, "p_value": ols.pvalues}).sort_values("p_value")
print("Smallest p-values (explanatory OLS):")
print(coef_table.head(15).to_string())

pred_test = ols.predict(X_test_df)
mae = (y_test - pred_test).abs().mean()
print("\nHoldout MAE (OLS):", round(float(mae), 3))


Smallest p-values (explanatory OLS):
                                           coef       p_value
const                                775.950156  1.674281e-07
cat__channel_source_PartnerReferral -267.577033  2.663948e-02
cat__channel_source_Event           -125.810924  2.062258e-01
cat__channel_source_Direct          -114.297712  2.581718e-01
cat__channel_source_SocialMedia      -82.544400  4.126711e-01
cat__campaign_name_GivingTuesday     117.198556  5.588428e-01
num__is_recurring_num                -38.312282  5.818940e-01
cat__campaign_name_Summer of Safety  -47.183661  7.929114e-01
cat__campaign_name_Year-End Hope     -20.948676  8.955147e-01
cat__campaign_name_Unknown            11.146890  9.353378e-01

Holdout MAE (OLS): 611.504


In [52]:
ols_hc1 = sm.OLS(y_train, X_train_df).fit(cov_type="HC1")
print(ols_hc1.summary())

coef = ols_hc1.params.drop("const", errors="ignore")
ci = ols_hc1.conf_int().loc[coef.index]
coef = coef.sort_values()
ci = ci.loc[coef.index]

plt.figure(figsize=(8, 4.5))
plt.errorbar(
    coef.values,
    coef.index,
    xerr=[coef.values - ci[0].values, ci[1].values - coef.values],
    fmt="o",
)
plt.axvline(0, color="gray", linestyle="--")
plt.title("OLS (HC1): coefficients with 95% CI")
plt.xlabel("Coefficient")
plt.tight_layout()
plt.close()


                            OLS Regression Results                            
Dep. Variable:        estimated_value   R-squared:                       0.021
Model:                            OLS   Adj. R-squared:                 -0.006
Method:                 Least Squares   F-statistic:                    0.9775
Date:                Wed, 08 Apr 2026   Prob (F-statistic):              0.458
Time:                        16:51:32   Log-Likelihood:                -2635.2
No. Observations:                 336   AIC:                             5290.
Df Residuals:                     326   BIC:                             5328.
Df Model:                           9                                         
Covariance Type:                  HC1                                         
                                          coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------
co

In [53]:
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_df.drop(columns=["const"], errors="ignore"), y_train)
X_test_nr = X_test_df.drop(columns=["const"], errors="ignore")
pred_ridge = ridge.predict(X_test_nr)
mae_ridge = mean_absolute_error(y_test, pred_ridge)
ridge_coef = pd.Series(ridge.coef_, index=X_test_nr.columns).sort_values(key=lambda s: s.abs(), ascending=False)
print("Ridge holdout MAE:", round(float(mae_ridge), 3))
print("\nTop |coef| (Ridge):")
print(ridge_coef.head(12).to_string())


Ridge holdout MAE: 611.242

Top |coef| (Ridge):
cat__channel_source_PartnerReferral   -255.036964
cat__channel_source_Event             -118.184610
cat__campaign_name_GivingTuesday       108.369964
cat__channel_source_Direct            -106.709410
cat__channel_source_SocialMedia        -75.369826
cat__campaign_name_Summer of Safety    -48.956592
num__is_recurring_num                  -37.934773
cat__campaign_name_Year-End Hope       -23.636451
cat__campaign_name_Unknown               8.686117


In [54]:
threshold = y_train.median()
y_train_high = (y_train >= threshold).astype(int)
y_test_high = (y_test >= threshold).astype(int)
X_train_log = X_train_df.drop(columns=["const"], errors="ignore")

logit = LogisticRegression(max_iter=2000)
logit.fit(X_train_log, y_train_high)

proba = logit.predict_proba(X_test_nr)[:, 1]
pred_cls = (proba >= 0.5).astype(int)
print("Median split threshold:", round(float(threshold), 3))
print("Accuracy:", round(float(accuracy_score(y_test_high, pred_cls)), 3))
print("ROC-AUC:", round(float(roc_auc_score(y_test_high, proba)), 3))

fpr, tpr, _ = roc_curve(y_test_high, proba)
plt.figure(figsize=(6, 4))
plt.plot(fpr, tpr, label=f"ROC AUC = {roc_auc_score(y_test_high, proba):.3f}")
plt.plot([0, 1], [0, 1], "--", color="gray")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("High-value gift vs not (logistic)")
plt.legend()
plt.tight_layout()
plt.close()


Median split threshold: 511.725
Accuracy: 0.571
ROC-AUC: 0.575


### Textbook Ch. 13 / Ch. 15: Classification metrics (high vs not)


In [55]:
from sklearn.metrics import classification_report, precision_recall_fscore_support

print(classification_report(y_test_high, pred_cls, target_names=["below_median", "above_median"], digits=3))

_p, _r, _f1, _ = precision_recall_fscore_support(y_test_high, pred_cls, average="binary", zero_division=0)
print(f"\nBinary average — precision: {_p:.3f}, recall: {_r:.3f}, F1: {_f1:.3f}")
print(
    "\nBusiness read (Ch. 15): A false positive flags a small gift as 'high' (wasted attention); "
    "a false negative misses a large gift (missed stewardship). Tune threshold if costs are asymmetric."
)


              precision    recall  f1-score   support

below_median      0.564     0.537     0.550        41
above_median      0.578     0.605     0.591        43

    accuracy                          0.571        84
   macro avg      0.571     0.571     0.570        84
weighted avg      0.571     0.571     0.571        84


Binary average — precision: 0.578, recall: 0.605, F1: 0.591

Business read (Ch. 15): A false positive flags a small gift as 'high' (wasted attention); a false negative misses a large gift (missed stewardship). Tune threshold if costs are asymmetric.


In [56]:
tree = DecisionTreeClassifier(max_depth=3, min_samples_leaf=15, random_state=42)
tree.fit(X_train_log, y_train_high)
print("Tree accuracy:", round(float(accuracy_score(y_test_high, tree.predict(X_test_nr))), 3))
print(export_text(tree, feature_names=list(X_train_log.columns)))

plt.figure(figsize=(12, 5))
plot_tree(tree, feature_names=list(X_train_log.columns), class_names=["low", "high"], filled=True, rounded=True, fontsize=8)
plt.title("Shallow tree (high-value classification)")
plt.close()


Tree accuracy: 0.56
|--- cat__channel_source_PartnerReferral <= 0.50
|   |--- cat__campaign_name_Year-End Hope <= 0.50
|   |   |--- cat__campaign_name_GivingTuesday <= 0.50
|   |   |   |--- class: 1
|   |   |--- cat__campaign_name_GivingTuesday >  0.50
|   |   |   |--- class: 1
|   |--- cat__campaign_name_Year-End Hope >  0.50
|   |   |--- cat__channel_source_Event <= 0.50
|   |   |   |--- class: 0
|   |   |--- cat__channel_source_Event >  0.50
|   |   |   |--- class: 0
|--- cat__channel_source_PartnerReferral >  0.50
|   |--- cat__campaign_name_Unknown <= 0.50
|   |   |--- class: 0
|   |--- cat__campaign_name_Unknown >  0.50
|   |   |--- class: 0



## 4. Evaluation & interpretation (holdout + CV + group errors)


In [57]:
ols_pred = ols_hc1.predict(X_test_df)
ridge_pred = ridge.predict(X_test_nr)

tree_reg = DecisionTreeRegressor(max_depth=3, min_samples_leaf=15, random_state=42)
tree_reg.fit(X_train_df.drop(columns=["const"], errors="ignore"), y_train)
tree_pred_reg = tree_reg.predict(X_test_nr)

eval_table = pd.DataFrame(
    [
        {"model": "OLS_HC1", "MAE": mean_absolute_error(y_test, ols_pred), "R2": r2_score(y_test, ols_pred)},
        {"model": "Ridge", "MAE": mean_absolute_error(y_test, ridge_pred), "R2": r2_score(y_test, ridge_pred)},
        {"model": "TreeReg(depth=3)", "MAE": mean_absolute_error(y_test, tree_pred_reg), "R2": r2_score(y_test, tree_pred_reg)},
    ]
).sort_values("MAE")

print("Holdout comparison:")
print(eval_table.to_string(index=False, float_format=lambda x: f"{x:,.3f}"))

kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_tree = -cross_val_score(
    DecisionTreeRegressor(max_depth=3, min_samples_leaf=15, random_state=42),
    X_train_df.drop(columns=["const"], errors="ignore"),
    y_train,
    cv=kf,
    scoring="neg_mean_absolute_error",
)
print("\n5-fold CV MAE (DecisionTreeRegressor):", round(float(cv_tree.mean()), 3), "+/-", round(float(cv_tree.std()), 3))

fair_df = pd.DataFrame(
    {
        "actual": y_test,
        "pred_ols": ols_pred,
        "pred_ridge": ridge_pred,
        "is_recurring": X_test_raw["is_recurring_num"],
    }
)
_rows = []
for _rec, _g in fair_df.groupby("is_recurring"):
    _rows.append(
        {
            "is_recurring": _rec,
            "ols_mae": mean_absolute_error(_g["actual"], _g["pred_ols"]),
            "ridge_mae": mean_absolute_error(_g["actual"], _g["pred_ridge"]),
        }
    )
print("\nMAE by recurrence (0/1):")
print(pd.DataFrame(_rows))

print(
    "\nBusiness read: errors are large relative to typical gifts — use for **directional** insight and portfolio mix, "
    "not for promising a specific amount to a donor."
)


Holdout comparison:
           model     MAE     R2
           Ridge 611.242 -0.064
         OLS_HC1 611.504 -0.066
TreeReg(depth=3) 611.684 -0.082

5-fold CV MAE (DecisionTreeRegressor): 480.616 +/- 67.672

MAE by recurrence (0/1):
   is_recurring     ols_mae   ridge_mae
0             0  675.489153  675.016216
1             1  521.923869  521.958314

Business read: errors are large relative to typical gifts — use for **directional** insight and portfolio mix, not for promising a specific amount to a donor.


### Textbook Ch. 9–11 & Ch. 15: Baseline, residuals, and time-based validation

- **Baseline (Ch. 15):** Compare models to predicting the **training mean** on the holdout set.
- **Residual plot (Ch. 9–11):** Patterns vs fitted values suggest missing structure (e.g. seasonality).
- **Time split (Ch. 5 / 15):** Sort by `donation_date` and hold out the **latest** gifts so validation mimics forecasting—not the same random split as above, so metrics will differ.


In [58]:
# --- Baseline vs OLS (same random split as Section 3–4) ---
baseline_pred = np.full(shape=y_test.shape, fill_value=y_train.mean())
_mae_base = mean_absolute_error(y_test, baseline_pred)
_mae_ols = mean_absolute_error(y_test, ols_pred)
print(f"Baseline MAE (train mean on test): {_mae_base:.3f}")
print(f"OLS (HC1) MAE on same split:       {_mae_ols:.3f}")
print("Beat baseline?" , _mae_ols < _mae_base)

# --- Residual plot (OLS HC1, holdout) ---
_resid = np.asarray(y_test) - np.asarray(ols_pred)
plt.figure(figsize=(6, 4))
plt.scatter(ols_pred, _resid, alpha=0.45)
plt.axhline(0, color="gray", linestyle="--")
plt.xlabel("Predicted value (OLS HC1)")
plt.ylabel("Residual (actual − predicted)")
plt.title("Residual plot — heteroskedasticity / missing structure (Ch. 9–11)")
plt.tight_layout()
plt.close()

# --- Time-ordered train/test (Ch. 5: temporal generalization) ---
def _donations_xy_from_raw(raw_part: pd.DataFrame):
    d = raw_part.copy()
    d["donation_date"] = pd.to_datetime(d["donation_date"], errors="coerce")
    d["donation_year"] = d["donation_date"].dt.year
    d["donation_month"] = d["donation_date"].dt.month
    d["has_social_referral"] = d["referral_post_id"].notna().astype(np.int8)
    d["is_recurring"] = (
        d["is_recurring"].astype(str).str.lower().map({"true": 1, "false": 0, "1": 1, "0": 0}).fillna(0).astype(np.int8)
    )
    y = pd.to_numeric(d["estimated_value"], errors="coerce")
    drop = ["donation_id", "supporter_id", "notes", "estimated_value", "donation_date", "amount", "referral_post_id"]
    X = d.drop(columns=[c for c in drop if c in d.columns], errors="ignore")
    m = y.notna()
    return X.loc[m].reset_index(drop=True), y.loc[m].reset_index(drop=True)

_raw = pd.read_csv(datasets_dir() / "donations.csv").dropna(subset=["estimated_value"])
_raw["donation_date"] = pd.to_datetime(_raw["donation_date"], errors="coerce")
_raw = _raw.sort_values("donation_date").reset_index(drop=True)
_n = len(_raw)
_cut = max(int(_n * 0.8), 80)
_tr_raw, _te_raw = _raw.iloc[:_cut], _raw.iloc[_cut:]
X_time_tr, y_time_tr = _donations_xy_from_raw(_tr_raw)
X_time_te, y_time_te = _donations_xy_from_raw(_te_raw)

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline

from pipeline_kit import build_preprocessor


def _coerce_bool_int(X):
    X = X.copy()
    for col in X.select_dtypes(include=["bool", "boolean"]).columns:
        X[col] = X[col].astype(np.int8)
    return X

X_time_tr = _coerce_bool_int(X_time_tr)
X_time_te = _coerce_bool_int(X_time_te)

_time_lin = Pipeline(
    [("prep", build_preprocessor(X_time_tr)), ("model", LinearRegression())]
)
_time_rf = Pipeline(
    [
        ("prep", build_preprocessor(X_time_tr)),
        ("model", RandomForestRegressor(n_estimators=200, random_state=42, min_samples_leaf=1)),
    ]
)
_time_lin.fit(X_time_tr, y_time_tr)
_time_rf.fit(X_time_tr, y_time_tr)
_pred_lin = _time_lin.predict(X_time_te)
_pred_rf = _time_rf.predict(X_time_te)
_base_time = np.full(shape=y_time_te.shape, fill_value=y_time_tr.mean())
print("\n--- Time-based holdout (last 20% by date) ---")
print("n_train:", len(y_time_tr), "| n_test:", len(y_time_te))
print("Baseline MAE:", round(float(mean_absolute_error(y_time_te, _base_time)), 3))
print("Linear MAE:  ", round(float(mean_absolute_error(y_time_te, _pred_lin)), 3))
print("RF MAE:      ", round(float(mean_absolute_error(y_time_te, _pred_rf)), 3))
print("Linear R2:   ", round(float(r2_score(y_time_te, _pred_lin)), 3))
print("RF R2:       ", round(float(r2_score(y_time_te, _pred_rf)), 3))


Baseline MAE (train mean on test): 609.537
OLS (HC1) MAE on same split:       611.504
Beat baseline? False

--- Time-based holdout (last 20% by date) ---
n_train: 336 | n_test: 84
Baseline MAE: 510.965
Linear MAE:   455.884
RF MAE:       413.582
Linear R2:    0.281
RF R2:        0.298


## 5. `pipeline_kit` — dual-track models (full feature set)

Uses `prepare_donations()` in `pipeline_kit.py`: **linear regression** (explanatory) and **random forest** (predictive). Artifacts are written under `ml-pipelines/artifacts/` for integration.

**Feature selection note:** `amount` is excluded in code to prevent leakage. Date parts and `has_social_referral` are included.


In [59]:
import joblib
from pathlib import Path

result = run_notebook_driver(NotebookConfig(pipeline_name="donations", seed=42, save_artifacts=True))
print("\nSaved artifact prefix: donations_*  under ml-pipelines/artifacts/")

# Optional: full sklearn pipelines for backend loading (same pattern as other *_predictive_model.joblib in artifacts/)
_bundle = {
    "explanatory_pipeline": result["explanatory"],
    "predictive_pipeline": result["predictive"],
    "task": result["task"],
    "meta": result["meta"],
}
joblib.dump(_bundle, Path("artifacts/donations_sklearn_pipelines.joblib"))
print("Wrote artifacts/donations_sklearn_pipelines.joblib")
print("Feature list for inference:", Path("artifacts/donations_model_schema.json").resolve())


=== donations ===
estimated_value (regression; mixed units in source data)
Shape X=(420, 9), task=regression, n=420

-- Metrics --
  explanatory_mae: 456.4538
  explanatory_r2: 0.2370
  predictive_mae: 455.2392
  predictive_r2: 0.1895

Top linear drivers (explanatory, |coef| proxy):


,feature,abs_coefficient
0,cat__donation_type_Monetary,314.775499
1,cat__impact_unit_pesos,314.775499
2,cat__impact_unit_hours,236.002594
3,cat__impact_unit_campaigns,160.119993
4,cat__donation_type_SocialMedia,160.119993
5,cat__donation_type_Time,148.601957
6,cat__campaign_name_GivingTuesday,142.164845
7,cat__channel_source_PartnerReferral,119.061643
8,cat__campaign_name_Summer of Safety,98.631716
9,cat__donation_type_Skills,87.400636



Top random forest importances (predictive):


,feature,importance
0,num__donation_month,0.230941
1,cat__donation_type_Monetary,0.156973
2,cat__impact_unit_pesos,0.155460
3,num__donation_year,0.101027
4,num__is_recurring,0.061925
5,cat__channel_source_Direct,0.049192
6,cat__donation_type_InKind,0.045354
7,cat__channel_source_Campaign,0.039607
8,cat__impact_unit_items,0.036290
9,cat__channel_source_Event,0.031833



Saved artifacts under C:\Users\abiga\IntextW2026\ml-pipelines\artifacts

Saved artifact prefix: donations_*  under ml-pipelines/artifacts/
Wrote artifacts/donations_sklearn_pipelines.joblib
Feature list for inference: C:\Users\abiga\IntextW2026\ml-pipelines\artifacts\donations_model_schema.json


## 6. Feature selection (Chapter 16) — donations-specific


In [60]:
# Reuse trained models from previous cell (no second fit)
from IPython.display import display

lin_imp = result["explanatory_coefs"]
rf_imp = result["predictive_importance"]

print("Metrics:", result["metrics"])

print("\nLinear (|coef|) top features:")
display(lin_imp)
print("\nRandomForest importance top features:")
display(rf_imp)

print(
    "\nJustification: we retain gift **type**, **impact unit**, **channel**, **campaign**, **recurrence**, **calendar**, "
    "and **referral** presence because they are actionable levers or structurally explain mix; we exclude raw `amount`."
)


Metrics: {'explanatory_mae': 456.4538139320165, 'explanatory_r2': 0.23696063453674043, 'predictive_mae': 455.239221207672, 'predictive_r2': 0.18949594526578817}

Linear (|coef|) top features:


,feature,abs_coefficient
0,cat__donation_type_Monetary,314.775499
1,cat__impact_unit_pesos,314.775499
2,cat__impact_unit_hours,236.002594
3,cat__impact_unit_campaigns,160.119993
4,cat__donation_type_SocialMedia,160.119993
5,cat__donation_type_Time,148.601957
6,cat__campaign_name_GivingTuesday,142.164845
7,cat__channel_source_PartnerReferral,119.061643
8,cat__campaign_name_Summer of Safety,98.631716
9,cat__donation_type_Skills,87.400636



RandomForest importance top features:


,feature,importance
0,num__donation_month,0.230941
1,cat__donation_type_Monetary,0.156973
2,cat__impact_unit_pesos,0.155460
3,num__donation_year,0.101027
4,num__is_recurring,0.061925
5,cat__channel_source_Direct,0.049192
6,cat__donation_type_InKind,0.045354
7,cat__channel_source_Campaign,0.039607
8,cat__impact_unit_items,0.036290
9,cat__channel_source_Event,0.031833



Justification: we retain gift **type**, **impact unit**, **channel**, **campaign**, **recurrence**, **calendar**, and **referral** presence because they are actionable levers or structurally explain mix; we exclude raw `amount`.


### Textbook Ch. 16: Permutation importance (predictive model)

Model-based importance can be **unstable** with correlated categoricals. **Permutation importance** on the held-out set shuffles each **raw input column** of `X` before the pipeline (one score per column of `result["X_test"]`), not per one-hot-expanded name from `get_feature_names_out()`.


In [61]:
from sklearn.inspection import permutation_importance

_perm = permutation_importance(
    result["predictive"],
    result["X_test"],
    result["y_test"],
    n_repeats=15,
    random_state=42,
    scoring="neg_mean_absolute_error",
)
# One importance value per *raw* column of X_test (sklearn shuffles each column before the pipeline).
_raw_names = list(result["X_test"].columns)
assert len(_raw_names) == len(_perm.importances_mean)
_perm_df = (
    pd.DataFrame(
        {"feature": _raw_names, "importance_mean": _perm.importances_mean, "importance_std": _perm.importances_std}
    )
    .sort_values("importance_mean", ascending=False)
    .reset_index(drop=True)
)
display(_perm_df)
print(
    "\nRead with caution (Ch. 16): higher values mean MAE gets worse when that column is shuffled (model relied on it). "
    "Categorical columns are permuted as a whole (all levels together), unlike RF split importances on one-hot columns."
)


,feature,importance_mean,importance_std
0,donation_type,74.112296,17.104161
1,impact_unit,73.659439,17.362417
2,donation_month,37.870258,16.288004
3,is_recurring,13.096760,9.436591
4,currency_code,0.000000,0.000000
5,has_social_referral,-2.640944,1.580544
6,channel_source,-8.444970,15.014971
7,campaign_name,-8.805902,7.670077
8,donation_year,-17.055440,13.807933



Read with caution (Ch. 16): higher values mean MAE gets worse when that column is shuffled (model relied on it). Categorical columns are permuted as a whole (all levels together), unlike RF split importances on one-hot columns.


## Textbook crosswalk (Ch. 1–17) — how this notebook maps

| Chapter theme | Where you see it in this notebook |
|---------------|-------------------------------------|
| Ch. 1 Process / framing | §1 stakeholder + predictive vs explanatory |
| Ch. 2–3 Data understanding | Data quality checks + missingness |
| Ch. 4–5 Prep / splitting | Prep pipelines; random split vs **time-based** split (§4b) |
| Ch. 6–8 EDA | Histograms, boxplots, correlations |
| Ch. 7 Reusable pipelines | `sklearn` `Pipeline` + `pipeline_kit` |
| Ch. 9–11 Linear / interpretation | OLS, HC1, Ridge, residuals |
| Ch. 12–13 Trees / classification | Shallow tree + logistic + **precision/recall/F1** |
| Ch. 14 Ensembles | Random forest (via `pipeline_kit`) |
| Ch. 15 Evaluation | MAE, R², CV, baseline, group MAE, error costs |
| Ch. 16 Feature selection | Domain drops (`amount`); RF + **permutation** importance |
| Ch. 17 Deployment / ops | §8 artifacts + monitoring ideas |

**Prediction vs explanation (Foreword):** Explanatory models prioritize interpretable associations; predictive models prioritize held-out error. Neither proves causation without design (§7).


## 7. Causal and relationship analysis (honest)

**What the models show:** Strong **association** between gift category (e.g., monetary vs. time), **impact unit**, and **channel** with recorded value. Recurrence and named campaigns also move the fitted lines.

**Causal claims:** We **cannot** conclude from observational gifts alone that “switching a donor to SocialMedia causes higher value.” Confounding (donor segment, seasonality, unobserved donor traits) is likely. The **explanatory** models summarize **historical association**; the **predictive** models prioritize rank-order fit on held-out rows.

**Recommendations (non-causal):**
- Compare **channel mix** and **campaign** performance in reporting; test small experiments before large budget shifts.
- Treat **social referral** (`has_social_referral`) as a hypothesis to validate with A/B or pre-post designs.

**Limitations:** Mixed units under one `estimated_value` target; interpret magnitudes as dataset-relative, not literal currency precision for every gift type.

**Multicollinearity:** In the linear model, `donation_type` and `impact_unit` overlap (e.g., monetary gifts use pesos), so some coefficient pairs mirror each other—read **direction and relative ranking**, not independent causal effects per dummy.


## 8. Deployment notes

**Artifacts:** `run_notebook_driver` writes `donations_model_metrics.csv`, `donations_top_features.csv`, `donations_top_explanatory_coefficients.csv`, and `donations_model_schema.json` under `ml-pipelines/artifacts/`. The notebook also saves `donations_sklearn_pipelines.joblib` (fitted explanatory + predictive `sklearn` pipelines).

**Web app:** Donation activity appears on admin analytics (`frontend/src/pages/AdminAnalytics.tsx`, `AdminDashboard.tsx`) via `/api/admin/data/donations` and OKR metrics. A production integration would load the saved **random forest** (or linear) pipeline from the joblib bundle in the backend and expose a **scenario** endpoint (“expected value given channel/campaign/type”) for internal staff only—mirroring patterns used for other pipelines in this repo.

**Optional local test:** load `donations_sklearn_pipelines.joblib` and call `.predict()` on `predictive_pipeline` with a DataFrame row matching `donations_model_schema.json` feature columns.

**`ml-service/app/main.py`:** Left unchanged—this file is the social-media analytics service only. Donations inference is exercised via the notebook artifacts and the standalone script `ml-pipelines/predict_donation_value.py` (run from repo root: `python ml-pipelines/predict_donation_value.py`, or pass `--json row.json`). A future team-owned API can load the same joblib without modifying `main.py`.


In [62]:
# Optional: persist manual preprocessor + Ridge for the same 3-feature story (small, interpretable)
import joblib

bundle = {
    "preprocessor": preprocessor,
    "model": ridge,
    "feature_columns": list(X_raw.columns),
    "target": "estimated_value",
    "notes": "Interpretable ridge on recurrence + campaign + channel; no amount feature.",
}
out = Path("artifacts/donations_manual_ridge_bundle.joblib")
out.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(bundle, out)
print("Wrote", out.resolve())


Wrote C:\Users\abiga\IntextW2026\ml-pipelines\artifacts\donations_manual_ridge_bundle.joblib
